# Silver — Pesquisa de Satisfação

**Entrada:** `case.bronze.tb_pesquisa_satisfacao`

**Saídas (schema `case.silver`):**

| Tabela | Granularidade | Para que serve |
|---|---|---|
| `tb_pesquisa_satisfacao` | 1 linha por pesquisa | Tabela tratada, com flags de qualidade. Nada é apagado. |
| `tb_pesquisa_satisfacao_chamado` | 1 linha por chamado | Consolidação (deduplicação) para a Gold: nota média, mínima, máxima e flags. |
| `tb_pesquisa_satisfacao_quarentena` | 1 linha por registro rejeitado | Linhas retiradas da tabela principal, com o motivo. |
| `tb_pesquisa_satisfacao_qualidade` | 1 linha por métrica | Métricas da execução (insumo do relatório de confiabilidade). |

**Regras de tratamento**
1. `id_pesquisa` e `id_chamado` são **inteiros (BIGINT)**, conforme decisão do grupo. Valor nulo, negativo, decimal ou texto vira nulo.
2. A nota é lida como texto e só então tipada. Vazio e textos como `NULL` ou `NaN` contam como nota ausente.
3. Nota ausente **não é imputada**: fica nula, com `flag_sem_nota`.
4. Nota fora de 1–5 (por exemplo -1, 0 e 6) **não é corrigida nem remapeada**: fica nula, com `flag_nota_fora_dominio`, e o valor original é preservado em `nota_original`.
5. `id_pesquisa` nulo ou repetido: vai para a quarentena (nas repetidas, fica a ingestão mais recente).
6. Chamado com mais de uma pesquisa: a tabela principal mantém todas, e a tabela `_chamado` entrega 1 linha por chamado (média das notas válidas), evitando duplicar linhas no join da Gold.
7. **Idempotência:** as tabelas são reconstruídas por completo a partir da Bronze (`overwrite`). Reexecutar produz o mesmo resultado.
8. **Portão de qualidade:** se as contagens não fecharem (Bronze = Silver + quarentena) ou houver chave duplicada, o notebook falha, e o job deve parar.

**Parâmetros** (widgets, para uso em Jobs): `catalog`, `bronze_schema`, `silver_schema`.
Ao final, o notebook devolve um resumo em JSON (`dbutils.notebook.exit`).

In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window


def parametro(nome, padrao):
    """Lê o widget/parâmetro do job. Fora do Databricks, usa o valor padrão."""
    try:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)
    except NameError:
        return padrao


CATALOG = parametro("catalog", "case")
BRONZE_SCHEMA = parametro("bronze_schema", "bronze")
SILVER_SCHEMA = parametro("silver_schema", "silver")

TABELA_ORIGEM = "tb_pesquisa_satisfacao"
TABELA_DESTINO = "tb_pesquisa_satisfacao"
TABELA_POR_CHAMADO = "tb_pesquisa_satisfacao_chamado"
TABELA_QUARENTENA = "tb_pesquisa_satisfacao_quarentena"
TABELA_QUALIDADE = "tb_pesquisa_satisfacao_qualidade"

FORMATO = "delta"
NOTA_MIN, NOTA_MAX = 1, 5
COLUNAS_ESPERADAS = {"id_pesquisa", "id_chamado", "nota_atendimento"}
VALORES_NULOS = ["NULL", "NAN", "N/A", "NA", "NONE", "-", "NULO"]


def fq(schema, tabela):
    """Nome totalmente qualificado. As crases protegem o catálogo `case` (palavra reservada em SQL)."""
    return f"`{CATALOG}`.{schema}.{tabela}"


spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.{SILVER_SCHEMA}")

In [0]:
def padronizar_colunas(df):
    """Nomes de coluna em minúsculas e sem espaços nas pontas."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower())
    return df


def texto_limpo(coluna):
    """Texto sem espaços nas pontas. Vazio, 'NULL', 'NaN' etc. viram nulo."""
    c = F.trim(F.col(coluna).cast("string"))
    return F.when(
        (F.length(c) == 0) | F.upper(c).isin(VALORES_NULOS),
        F.lit(None).cast("string"),
    ).otherwise(c)


def numero(coluna):
    """Texto -> DOUBLE. Retorna nulo para o que não for número finito (inclui NaN e infinito)."""
    num = F.expr(f"try_cast(`{coluna}` AS DOUBLE)")
    return F.when(~F.isnan(num) & (F.abs(num) < 9007199254740991), num)


def id_inteiro(coluna):
    """Texto -> BIGINT. Só aceita inteiro não negativo; o resto vira nulo."""
    num = numero(coluna)
    return F.when((num >= 0) & (num == F.floor(num)), num.cast("long"))


def tipar_e_sinalizar(df):
    df = padronizar_colunas(df)

    faltando = COLUNAS_ESPERADAS - set(df.columns)
    if faltando:
        raise ValueError(f"Colunas ausentes na Bronze: {faltando}. Encontradas: {df.columns}")

    df = (
        df.withColumn("id_pesquisa_original", texto_limpo("id_pesquisa"))
          .withColumn("id_chamado_original", texto_limpo("id_chamado"))
          .withColumn("nota_original", texto_limpo("nota_atendimento"))
          .withColumn("id_pesquisa", id_inteiro("id_pesquisa_original"))
          .withColumn("id_chamado", id_inteiro("id_chamado_original"))
          .withColumn("_nota_num", numero("nota_original"))
    )

    nao_inteira = F.col("_nota_num").isNotNull() & (F.col("_nota_num") != F.floor(F.col("_nota_num")))
    fora_dominio = F.col("_nota_num").isNotNull() & (
        (F.col("_nota_num") < NOTA_MIN) | (F.col("_nota_num") > NOTA_MAX)
    )
    nota_valida = F.col("_nota_num").isNotNull() & ~nao_inteira & ~fora_dominio

    df = (
        df.withColumn("flag_sem_nota", F.col("nota_original").isNull())
          .withColumn("flag_nota_nao_numerica", F.col("nota_original").isNotNull() & F.col("_nota_num").isNull())
          .withColumn("flag_nota_nao_inteira", nao_inteira)
          .withColumn("flag_nota_fora_dominio", fora_dominio)
          .withColumn("flag_id_chamado_invalido", F.col("id_chamado").isNull())
          .withColumn("nota_atendimento", F.when(nota_valida, F.col("_nota_num").cast("int")))
          .drop("_nota_num")
    )

    if "ingestion_datetime" in df.columns:
        df = df.withColumnRenamed("ingestion_datetime", "bronze_ingestion_datetime")
    else:
        df = df.withColumn("bronze_ingestion_datetime", F.lit(None).cast("timestamp"))

    return df


def separar_quarentena(df):
    """id_pesquisa nulo/inválido e id_pesquisa repetido vão para a quarentena, com o motivo."""
    sem_chave = df.filter(F.col("id_pesquisa").isNull()).withColumn(
        "motivo_quarentena", F.lit("id_pesquisa_nulo_ou_invalido")
    )
    com_chave = df.filter(F.col("id_pesquisa").isNotNull())

    w = Window.partitionBy("id_pesquisa").orderBy(
        F.col("bronze_ingestion_datetime").desc(),
        F.col("nota_original").desc_nulls_last(),
        F.col("id_chamado").desc_nulls_last(),
    )
    ranqueado = com_chave.withColumn("_rn", F.row_number().over(w))

    validos = ranqueado.filter("_rn = 1").drop("_rn")
    duplicados = (
        ranqueado.filter("_rn > 1").drop("_rn")
        .withColumn("motivo_quarentena", F.lit("duplicata_id_pesquisa"))
    )
    return validos, sem_chave.unionByName(duplicados)


def sinalizar_multiplas_pesquisas(df):
    w = Window.partitionBy("id_chamado")
    return (
        df.withColumn(
            "qtd_pesquisas_chamado",
            F.when(F.col("id_chamado").isNotNull(), F.count(F.lit(1)).over(w)),
        )
        .withColumn(
            "flag_chamado_multiplas_pesquisas",
            F.coalesce(F.col("qtd_pesquisas_chamado") > 1, F.lit(False)),
        )
    )


def consolidar_por_chamado(silver):
    """1 linha por chamado. A nota média considera só notas válidas; nada é removido da tabela principal."""
    return (
        silver.filter(F.col("id_chamado").isNotNull())
        .groupBy("id_chamado")
        .agg(
            F.count(F.lit(1)).alias("qtd_pesquisas"),
            F.count("nota_atendimento").alias("qtd_notas_validas"),
            F.round(F.avg("nota_atendimento"), 4).alias("nota_media"),
            F.min("nota_atendimento").alias("nota_min"),
            F.max("nota_atendimento").alias("nota_max"),
        )
        .withColumn("flag_sem_nota_valida", F.col("qtd_notas_validas") == 0)
        .withColumn(
            "flag_notas_divergentes",
            (F.col("qtd_notas_validas") > 1) & (F.col("nota_min") != F.col("nota_max")),
        )
    )


COLUNAS_SILVER = [
    "id_pesquisa", "id_chamado", "nota_atendimento", "nota_original",
    "flag_sem_nota", "flag_nota_nao_numerica", "flag_nota_nao_inteira",
    "flag_nota_fora_dominio", "flag_id_chamado_invalido",
    "qtd_pesquisas_chamado", "flag_chamado_multiplas_pesquisas",
    "bronze_ingestion_datetime",
]

COLUNAS_QUARENTENA = [
    "motivo_quarentena", "id_pesquisa_original", "id_chamado_original",
    "id_pesquisa", "id_chamado", "nota_original", "bronze_ingestion_datetime",
]


def gravar(df, tabela):
    """Reconstrução completa da tabela: reexecutar não duplica nada."""
    (
        df.withColumn("silver_processed_datetime", F.current_timestamp())
          .write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(fq(SILVER_SCHEMA, tabela))
    )


def validar_integridade(n_bronze, silver, quarentena, por_chamado):
    """Portão de qualidade: se algo não fechar, o notebook falha e o job para."""
    n_silver, n_quarentena = silver.count(), quarentena.count()
    erros = []

    if n_bronze != n_silver + n_quarentena:
        erros.append(f"Contagem não fecha: bronze={n_bronze}, silver={n_silver}, quarentena={n_quarentena}")

    if silver.select("id_pesquisa").distinct().count() != n_silver:
        erros.append("id_pesquisa duplicado na Silver")

    fora = silver.filter(
        (F.col("nota_atendimento") < NOTA_MIN) | (F.col("nota_atendimento") > NOTA_MAX)
    ).count()
    if fora:
        erros.append(f"{fora} nota(s) fora de {NOTA_MIN}-{NOTA_MAX} na coluna nota_atendimento")

    chamados = silver.filter("id_chamado IS NOT NULL").select("id_chamado").distinct().count()
    if por_chamado.count() != chamados or por_chamado.select("id_chamado").distinct().count() != chamados:
        erros.append("A tabela por chamado não tem exatamente 1 linha por id_chamado")

    if erros:
        raise ValueError("Falha na validação de integridade: " + " | ".join(erros))


def calcular_qualidade(n_bronze, silver, quarentena, por_chamado):
    num_original = F.expr("try_cast(nota_original AS DOUBLE)")
    a = silver.agg(
        F.count(F.lit(1)).alias("linhas_silver"),
        F.sum(F.col("nota_atendimento").isNotNull().cast("int")).alias("notas_validas"),
        F.sum(F.col("flag_sem_nota").cast("int")).alias("sem_nota"),
        F.sum(F.col("flag_nota_nao_numerica").cast("int")).alias("nota_nao_numerica"),
        F.sum(F.col("flag_nota_nao_inteira").cast("int")).alias("nota_nao_inteira"),
        F.sum(F.col("flag_nota_fora_dominio").cast("int")).alias("nota_fora_dominio"),
        F.sum((num_original < NOTA_MIN).cast("int")).alias("fora_escala_abaixo_do_minimo"),
        F.sum((num_original > NOTA_MAX).cast("int")).alias("fora_escala_acima_do_maximo"),
        F.sum(F.col("flag_id_chamado_invalido").cast("int")).alias("id_chamado_invalido"),
        F.sum(F.col("flag_chamado_multiplas_pesquisas").cast("int")).alias("linhas_em_chamados_com_varias_pesquisas"),
    ).collect()[0].asDict()
    a = {k: float(v or 0) for k, v in a.items()}

    c = por_chamado.agg(
        F.count(F.lit(1)).alias("chamados_distintos"),
        F.sum((F.col("qtd_pesquisas") > 1).cast("int")).alias("chamados_com_varias_pesquisas"),
        F.sum(F.col("flag_notas_divergentes").cast("int")).alias("chamados_com_notas_divergentes"),
        F.sum(F.col("flag_sem_nota_valida").cast("int")).alias("chamados_sem_nota_valida"),
    ).collect()[0].asDict()
    c = {k: float(v or 0) for k, v in c.items()}

    q = {r["motivo_quarentena"]: float(r["count"]) for r in quarentena.groupBy("motivo_quarentena").count().collect()}

    metricas = {"linhas_bronze": float(n_bronze), **a, **c}
    metricas["quarentena_duplicata_id_pesquisa"] = q.get("duplicata_id_pesquisa", 0.0)
    metricas["quarentena_id_pesquisa_nulo_ou_invalido"] = q.get("id_pesquisa_nulo_ou_invalido", 0.0)
    total = a["linhas_silver"]
    metricas["pct_notas_validas"] = round(100 * a["notas_validas"] / total, 2) if total else 0.0
    metricas["pct_sem_nota"] = round(100 * a["sem_nota"] / total, 2) if total else 0.0

    return spark.createDataFrame(list(metricas.items()), "metrica string, valor double"), metricas

In [0]:
def executar():
    df_bronze = spark.table(fq(BRONZE_SCHEMA, TABELA_ORIGEM))
    n_bronze = df_bronze.count()

    validos, quarentena = separar_quarentena(tipar_e_sinalizar(df_bronze))
    silver = sinalizar_multiplas_pesquisas(validos).select(*COLUNAS_SILVER)

    gravar(silver, TABELA_DESTINO)
    gravar(quarentena.select(*COLUNAS_QUARENTENA), TABELA_QUARENTENA)

    silver_tbl = spark.table(fq(SILVER_SCHEMA, TABELA_DESTINO))
    quarentena_tbl = spark.table(fq(SILVER_SCHEMA, TABELA_QUARENTENA))

    gravar(consolidar_por_chamado(silver_tbl), TABELA_POR_CHAMADO)
    por_chamado_tbl = spark.table(fq(SILVER_SCHEMA, TABELA_POR_CHAMADO))

    validar_integridade(n_bronze, silver_tbl, quarentena_tbl, por_chamado_tbl)

    df_qualidade, metricas = calcular_qualidade(n_bronze, silver_tbl, quarentena_tbl, por_chamado_tbl)
    gravar(df_qualidade, TABELA_QUALIDADE)

    return {k: int(v) if float(v).is_integer() else v for k, v in metricas.items()}


resumo = executar()
print(json.dumps(resumo, ensure_ascii=False))

try:
    dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False))
except NameError:
    pass